# Chapitre 17 · Apprendre à obéir

Notebook du chapitre 17 de *Construire un LLM de zéro*. Ton GPT des fables (chapitre 10) sait compléter du texte. Ici, tu le transformes en assistant qui répond aux questions, et tu découvres pourquoi ça casse quand on s'y prend mal.

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et exécutable de bout en bout. Lis, exécute, modifie pour voir. À la fin, la section **Exercices** : quatre défis à trous, du plus simple (●) au plus costaud (●●●), validés par des `assert`.

Au programme, dans l'ordre :

1. **Le fossé** : le modèle de base complète, il n'obéit pas (démonstration).
2. **Le format d'instruction** : tokens spéciaux, template, et masquage de la loss sur le prompt.
3. **SFT** : le même modèle, avant / après, sur les mêmes questions.
4. **Deux cas qui échouent** : le masque oublié (le modèle se parle à lui-même) et l'oubli catastrophique (la perplexité fables explose).
5. **LoRA de zéro** : la correction de rang faible W + (alpha/r)·B·A, écrite à la main, et l'adaptateur qu'on branche et débranche.
6. **Un jouet 4 bits** : l'idée de QLoRA, mesurée sur nos poids.

Tout tourne sur CPU en quelques minutes (quatre entraînements courts). La cellule QLoRA réelle est réservée à Colab GPU et se saute proprement ailleurs.

## 0. Le point de départ : le GPT des fables (chapitre 10)

On récupère le corpus des 30 fables, le tokeniseur caractère et le GPT du chapitre 10, avec un seul ajout : **trois tokens spéciaux** (`<|user|>`, `<|assistant|>`, `<|fin|>`) réservés pour le dialogue. Ils existent dès maintenant dans le vocabulaire, mais le pré-entraînement ne les verra jamais : exactement comme les vrais labos, qui réservent les tokens de conversation dans le tokeniseur avant même le pré-entraînement.

In [1]:
import time, math, random, copy
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
print("torch", torch.__version__)

torch 2.12.1


In [2]:
corpus = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêter
Quelque grain pour subsister
Jusqu'à la saison nouvelle.
Je vous paierai, lui dit-elle,
Avant l'oût, foi d'animal,
Intérêt et principal.
La fourmi n'est pas prêteuse :
C'est là son moindre défaut.
Que faisiez-vous au temps chaud ?
Dit-elle à cette emprunteuse. -
Nuit et jour à tout venant
Je chantais, ne vous déplaise. -
Vous chantiez, j'en suis fort aise !
Eh bien ! dansez maintenant.


LE CORBEAU ET LE RENARD
Maître corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Hé ! bonjour, monsieur du corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le phénix des hôtes de ces bois.
À ces mots le corbeau ne se sent pas de joie ;
Et, pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le renard s'en saisit, et dit : Mon bon monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute :
Cette leçon vaut bien un fromage, sans doute.
Le corbeau, honteux et confus,
Jura, mais un peu tard, qu'on ne l'y prendrait plus.


LA GRENOUILLE QUI SE VEUT FAIRE AUSSI GROSSE QUE LE BŒUF
Une grenouille vit un bœuf
Qui lui sembla de belle taille.
Elle, qui n'était pas grosse en tout comme un œuf,
Envieuse, s'étend, et s'enfle, et se travaille
Pour égaler l'animal en grosseur ;
Disant : Regardez bien, ma sœur ;
Est-ce assez ? dites-moi ; n'y suis-je point encore ? -
Nenni. - M'y voici donc ? - Point du tout. - M'y voilà ? -
Vous n'en approchez point. La chétive pécore
S'enfla si bien qu'elle creva.
Le monde est plein de gens qui ne sont pas plus sages :
Tout bourgeois veut bâtir comme les grands seigneurs,
Tout petit prince a des ambassadeurs,
Tout marquis veut avoir des pages.


LE LOUP ET LE CHIEN
Un loup n'avait que les os et la peau,
Tant les chiens faisaient bonne garde.
Ce loup rencontre un dogue aussi puissant que beau,
Gras, poli, qui s'était fourvoyé par mégarde.
L'attaquer, le mettre en quartiers,
Sire loup l'eût fait volontiers :
Mais il fallait livrer bataille ;
Et le mâtin était de taille
À se défendre hardiment.
Le loup donc l'aborde humblement,
Entre en propos, et lui fait compliment
Sur son embonpoint, qu'il admire.
Il ne tiendra qu'à vous, beau sire,
D'être aussi gras que moi, lui repartit le chien.
Quittez les bois, vous ferez bien :
Vos pareils y sont misérables,
Cancres, hères et pauvres diables,
Dont la condition est de mourir de faim.
Car, quoi ! rien d'assuré ! point de franche lippée !
Tout à la pointe de l'épée !
Suivez-moi, vous aurez un bien meilleur destin.
Le loup reprit : Que me faudra-t-il faire ?
Presque rien, dit le chien : donner la chasse aux gens
Portants bâtons, et mendiants ;
Flatter ceux du logis, à son maître complaire ;
Moyennant quoi votre salaire
Sera force reliefs de toutes les façons,
Os de poulets, os de pigeons ;
Sans parler de mainte caresse.
Le loup déjà se forge une félicité
Qui le fait pleurer de tendresse.
Chemin faisant il vit le cou du chien pelé.
Qu'est-ce là ? lui dit-il. - Rien. - Quoi ! rien ! - Peu de chose. -
Mais encor ? - Le collier dont je suis attaché
De ce que vous voyez est peut-être la cause.
Attaché ! dit le loup : vous ne courez donc pas
Où vous voulez ? - Pas toujours ; mais qu'importe ?
Il importe si bien, que de tous vos repas
Je ne veux en aucune sorte,
Et ne voudrais pas même à ce prix d'un trésor.
Cela dit, maître loup s'enfuit, et court encor.


LA BESACE
Jupiter dit un jour : Que tout ce qui respire
S'en vienne comparaître aux pieds de ma grandeur :
Si dans son composé quelqu'un trouve à redire,
Il peut le déclarer sans peur ;
Je mettrai remède à la chose.
Venez, singe ; parlez le premier, et pour cause :
Voyez ces animaux, faites comparaison
De leurs beautés avec les vôtres.
Êtes-vous satisfait ? - Moi, dit-il ; pourquoi non ?
N'ai-je pas quatre pieds aussi bien que les autres ?
Mon portrait jusqu'ici ne m'a rien reproché :
Mais pour mon frère l'ours, on ne l'a qu'ébauché ;
Jamais, s'il me veut croire, il ne se fera peindre.
L'ours venant là-dessus, on crut qu'il s'allait plaindre.
Tant s'en faut : de sa forme il se loua très-fort ;
Glosa sur l'éléphant, dit qu'on pourrait encor
Ajouter à sa queue, ôter à ses oreilles ;
Que c'était une masse informe et sans beauté.
L'éléphant étant écouté,
Tout sage qu'il était, dit des choses pareilles :
Il jugea qu'à son appétit
Dame baleine était trop grosse.
Dame fourmi trouva le ciron trop petit,
Se croyant, pour elle, un colosse.
Jupin les renvoya s'étant censurés tous,
Du reste, contents d'eux. Mais, parmi les plus fous,
Notre espèce excella ; car, tout ce que nous sommes,
Lynx envers nos pareils, et taupes envers nous,
Nous nous pardonnons tout, et rien aux autres hommes :
On se voit d'un autre œil qu'on ne voit son prochain.
Le fabricateur souverain
Nous créa besaciers tous de même manière,
Tant ceux du temps passé que du temps d'aujourd'hui :
Il fit pour nos défauts la poche de derrière,
Et celle de devant pour les défauts d'autrui.


LE LOUP ET L'AGNEAU
La raison du plus fort est toujours la meilleure :
Nous l'allons montrer tout à l'heure.
Un agneau se désaltérait
Dans le courant d'une onde pure.
Un loup survint à jeun, qui cherchait aventure,
Et que la faim en ces lieux attirait.
Qui te rend si hardi de troubler mon breuvage ?
Dit cet animal plein de rage :
Tu seras châtié de ta témérité.
Sire, répond l'agneau, que Votre Majesté
Ne se mette pas en colère ;
Mais plutôt qu'elle considère
Que je me vas désaltérant
Dans le courant,
Plus de vingt pas au-dessous d'elle ;
Et que, par conséquent, en aucune façon
Je ne puis troubler sa boisson.
Tu la troubles ! reprit cette bête cruelle ;
Et je sais que de moi tu médis l'an passé.
Comment l'aurais-je fait, si je n'étais pas né ?
Reprit l'agneau : je tette encore ma mère. -
Si ce n'est toi, c'est donc ton frère. -
Je n'en ai point. - C'est donc quelqu'un des tiens ;
Car vous ne m'épargnez guère,
Vous, vos bergers et vos chiens.
On me l'a dit : il faut que je me venge.
Là-dessus, au fond des forêts
Le loup l'emporte, et puis le mange,
Sans autre forme de procès.


LA MORT ET LE BÛCHERON
Un pauvre bucheron, tout couvert de ramée,
Sous le faix du fagot aussi bien que des ans,
Gémissant et courbé, marchait à pas pesants,
Et tâchait de gagner sa chaumine enfumée.
Enfin, n'en pouvant plus d'effort et de douleur,
Il met bas son fagot, il songe à son malheur.
Quel plaisir a-t-il eu depuis qu'il est au monde ?
En est-il un plus pauvre en la machine ronde ?
Point de pain quelquefois, et jamais de repos :
Sa femme, ses enfants, les soldats, les impôts,
Le créancier, et la corvée,
Lui font d'un malheureux la peinture achevée.
Il appelle la Mort. Elle vient sans tarder,
Lui demande ce qu'il faut faire.
C'est, dit-il, afin de m'aider
À recharger ce bois ; tu ne tarderas guère.
Le trépas vient tout guérir ;
Mais ne bougeons d'où nous sommes :
Plutôt souffrir que mourir,
C'est la devise des hommes.


LE RENARD ET LA CIGOGNE
Compère le renard se mit un jour en frais,
Et retint à dîner commère la cigogne.
Le régal fut petit et sans beaucoup d'apprêts :
Le galant, pour toute besogne,
Avait un brouet clair ; il vivait chichement.
Ce brouet fut par lui servi sur une assiette :
La cigogne au long bec n'en put attraper miette ;
Et le drôle eut lapé le tout en un moment.
Pour se venger de cette tromperie,
À quelque temps de là la cigogne le prie.
Volontiers, lui dit-il ; car avec mes amis
Je ne fais point cérémonie.
À l'heure dite, il courut au logis
De la cigogne son hôtesse ;
Loua très-fort sa politesse ;
Trouva le dîner cuit à point :
Bon appétit surtout ; renards n'en manquent point.
Il se réjouissait à l'odeur de la viande
Mise en menus morceaux, et qu'il croyait friande.
On servit, pour l'embarrasser,
En un vase à long col et d'étroite embouchure.
Le bec de la cigogne y pouvait bien passer ;
Mais le museau du sire était d'autre mesure.
Il lui fallut à jeun retourner au logis,
Honteux comme un renard qu'une poule aurait pris,
Serrant la queue, et portant bas l'oreille.
Trompeurs, c'est pour vous que j'écris :
Attendez-vous à la pareille.


LE CHÊNE ET LE ROSEAU
Le chêne un jour dit au roseau :
Vous avez bien sujet d'accuser la nature ;
Un roitelet pour vous est un pesant fardeau :
Le moindre vent qui d'aventure
Fait rider la face de l'eau,
Vous oblige à baisser la tête ;
Cependant que mon front, au Caucase pareil,
Non content d'arrêter les rayons du soleil,
Brave l'effort de la tempête.
Tout vous est aquilon, tout me semble zéphyr.
Encor si vous naissiez à l'abri du feuillage
Dont je couvre le voisinage,
Vous n'auriez pas tant à souffrir,
Je vous défendrais de l'orage :
Mais vous naissez le plus souvent
Sur les humides bords des royaumes du vent.
La nature envers vous me semble bien injuste.
Votre compassion, lui répondit l'arbuste,
Part d'un bon naturel ; mais quittez ce souci :
Les vents me sont moins qu'à vous redoutables ;
Je plie et ne romps pas. Vous avez jusqu'ici
Contre leurs coups épouvantables
Résisté sans courber le dos ;
Mais attendons la fin. Comme il disait ces mots,
Du bout de l'horizon accourt avec furie
Le plus terrible des enfants
Que le Nord eût portés jusque-là dans ses flancs.
L'arbre tient bon ; le roseau plie.
Le vent redouble ses efforts,
Et fait si bien qu'il déracine
Celui de qui la tête au ciel était voisine,
Et dont les pieds touchaient à l'empire des morts.


LE LION ET LE RAT
Il faut, autant qu'on peut, obliger tout le monde :
On a souvent besoin d'un plus petit que soi.
De cette vérité deux fables feront foi ;
Tant la chose en preuves abonde.
Entre les pattes d'un lion
Un rat sortit de terre assez à l'étourdie.
Le roi des animaux, en cette occasion,
Montra ce qu'il était, et lui donna la vie.
Ce bienfait ne fut pas perdu.
Quelqu'un aurait-il jamais cru
Qu'un lion d'un rat eût affaire ?
Cependant il advint qu'au sortir des forêts
Ce lion fut pris dans des rets,
Dont ses rugissements ne le purent défaire.
Sire rat accourut, et fit tant par ses dents
Qu'une maille rongée emporta tout l'ouvrage.
Patience et longueur de temps
Font plus que force ni que rage.


LA COLOMBE ET LA FOURMI
L'autre exemple est tiré d'animaux plus petits.
Le long d'un clair ruisseau buvait une colombe,
Quand sur l'eau se penchant une fourmis y tombe ;
Et dans cet océan on eût vu la fourmis
S'efforcer, mais en vain, de regagner la rive.
La colombe aussitôt usa de charité :
Un brin d'herbe dans l'eau par elle étant jeté,
Ce fut un promontoire où la fourmis arrive.
Elle se sauve. Et là-dessus
Passe un certain croquant qui marchait les pieds nus :
Ce croquant, par hasard, avait une arbalète.
Dès qu'il voit l'oiseau de Vénus,
Il le croit en son pot, et déjà lui fait fête.
Tandis qu'à le tuer mon villageois s'apprête,
La fourmi le pique au talon.
Le vilain retourne la tête :
La colombe l'entend, part, et tire de long.
Le souper du croquant avec elle s'envole :
Point de pigeon pour une obole.


LE LIÈVRE ET LA TORTUE
Rien ne sert de courir ; il faut partir à point :
Le lièvre et la tortue en sont un témoignage.
Gageons, dit celle-ci, que vous n'atteindrez point
Sitôt que moi ce but. Sitôt ! êtes-vous sage ?
Repartit l'animal léger :
Ma commère, il vous faut purger
Avec quatre grains d'ellébore.
- Sage ou non, je parie encore.
Ainsi fut fait ; et de tous deux
On mit près du but les enjeux.
Savoir quoi, ce n'est pas l'affaire,
Ni de quel juge l'on convint.
Notre lièvre n'avait que quatre pas à faire ;
J'entends de ceux qu'il fait lorsque, près d'être atteint,
Il s'éloigne des chiens, les renvoie aux calendes,
Et leur fait arpenter les landes.
Ayant, dis-je, du temps de reste pour brouter,
Pour dormir, et pour écouter
D'où vient le vent, il laisse la tortue
Aller son train de sénateur.
Elle part, elle s'évertue ;
Elle se hâte avec lenteur.
Lui cependant méprise une telle victoire,
Tient la gageure à peu de gloire,
Croit qu'il y va de son honneur
De partir tard. Il broute, il se repose ;
Il s'amuse à toute autre chose
Qu'à la gageure. À la fin, quand il vit
Que l'autre touchait presque au bout de la carrière,
Il partit comme un trait ; mais les élans qu'il fit
Furent vains : la tortue arriva la première.
Eh bien ! lui cria-t-elle, avais-je pas raison ?
De quoi vous sert votre vitesse ?
Moi l'emporter ! et que serait-ce
Si vous portiez une maison ?


LE RENARD ET LES RAISINS
Certain renard gascon, d'autres disent normand,
Mourant presque de faim, vit au haut d'une treille
Des raisins, mûrs apparemment,
Et couverts d'une peau vermeille.
Le galant en eût fait volontiers un repas ;
Mais comme il n'y pouvait atteindre :
Ils sont trop verts, dit-il, et bons pour des goujats.
Fit-il pas mieux que de se plaindre ?


LE HÉRON
Un jour, sur ses longs pieds, allait je ne sais où
Le héron au long bec emmanché d'un long cou :
Il côtoyait une rivière.
L'onde était transparente ainsi qu'aux plus beaux jours ;
Ma commère la carpe y faisait mille tours
Avec le brochet son compère.
Le héron en eût fait aisément son profit :
Tous approchaient du bord ; l'oiseau n'avait qu'à prendre.
Mais il crut mieux faire d'attendre
Qu'il eût un peu plus d'appétit :
Il vivait de régime, et mangeait à ses heures.
Après quelques moments l'appétit vint : l'oiseau,
S'approchant du bord, vit sur l'eau
Des tanches qui sortaient du fond de ces demeures.
Le mets ne lui plut pas, il s'attendait à mieux,
Et montrait un goût dédaigneux
Comme le rat du bon Horace.
Moi, des tanches ! dit-il ; moi, héron, que je fasse
Une si pauvre chère ! Et pour qui me prend-on ?
La tanche rebutée, il trouva du goujon.
Du goujon ! c'est bien là le dîner d'un héron !
J'ouvrirais pour si peu le bec ! aux dieux ne plaise !
Il l'ouvrit pour bien moins : tout alla de façon
Qu'il ne vit plus aucun poisson.
La faim le prit : il fut tout heureux et tout aise
De rencontrer un limaçon.
Ne soyons pas si difficiles :
Les plus accommodants, ce sont les plus habiles ;
On hasarde de perdre en voulant trop gagner.
Gardez-vous de rien dédaigner,
Surtout quand vous avez à peu près votre compte.
Bien des gens y sont pris. Ce n'est pas aux hérons
Que je parle : écoutez, humains, un autre conte :
Vous verrez que chez vous j'ai puisé ces leçons.


LA LAITIÈRE ET LE POT AU LAIT
Perrette, sur sa tête ayant un pot au lait
Bien posé sur un coussinet,
Prétendait arriver sans encombre à la ville.
Légère et court vêtue, elle allait à grands pas,
Ayant mis ce jour-là, pour être plus agile,
Cotillon simple et souliers plats.
Notre laitière ainsi troussée
Comptait déjà dans sa pensée
Tout le prix de son lait ; en employait l'argent ;
Achetait un cent d'œufs ; faisait triple couvée :
La chose allait à bien par son soin diligent.
Il m'est, disait-elle, facile
D'élever des poulets autour de ma maison ;
Le renard sera bien habile
S'il ne m'en laisse assez pour avoir un cochon.
Le porc à s'engraisser coûtera peu de son ;
Il était, quand je l'eus, de grosseur raisonnable :
J'aurai, le revendant, de l'argent bel et bon.
Et qui m'empêchera de mettre en notre étable,
Vu le prix dont il est, une vache et son veau,
Que je verrai sauter au milieu du troupeau ?
Perrette là-dessus saute aussi, transportée :
Le lait tombe ; adieu veau, vache, cochon, couvée.
La dame de ces biens, quittant d'un œil marri
Sa fortune ainsi répandue,
Va s'excuser à son mari,
En grand danger d'être battue.
Le récit en farce en fut fait ;
On l'appela le Pot au lait.
Quel esprit ne bat la campagne ?
Qui ne fait châteaux en Espagne ?
Picrochole, Pyrrhus, la laitière, enfin tous,
Autant les sages que les fous.
Chacun songe en veillant ; il n'est rien de plus doux
Une flatteuse erreur emporte alors nos âmes ;
Tout le bien du monde est à nous,
Tous les honneurs, toutes les femmes.
Quand je suis seul, je fais au plus brave un défi ;
Je m'écarte, je vais détrôner le sophi ;
On m'élit roi, mon peuple m'aime ;
Les diadèmes vont sur ma tête pleuvant :
Quelque accident fait-il que je rentre en moi-même ;
Je suis Gros-Jean comme devant.


LE COCHE ET LA MOUCHE
Dans un chemin montant, sablonneux, malaisé,
Et de tous les côtés au soleil exposé,
Six forts chevaux tiraient un coche.
Femmes, moine, vieillards, tout était descendu :
L'attelage suait, soufflait, était rendu.
Une mouche survient, et des chevaux s'approche,
Prétend les animer par son bourdonnement,
Pique l'un, pique l'autre, et pense à tout moment
Qu'elle fait aller la machine,
S'assied sur le timon, sur le nez du cocher.
Aussitôt que le char chemine,
Et qu'elle voit les gens marcher,
Elle s'en attribue uniquement la gloire,
Va, vient, fait l'empressée : il semble que ce soit
Un sergent de bataille allant en chaque endroit
Faire avancer ses gens et hâter la victoire.
La mouche, en ce commun besoin,
Se plaint qu'elle agit seule, et qu'elle a tout le soin ;
Qu'aucun n'aide aux chevaux à se tirer d'affaire.
Le moine disait son bréviaire :
Il prenait bien son temps ! une femme chantait :
C'était bien de chansons qu'alors il s'agissait !
Dame mouche s'en va chanter à leurs oreilles,
Et fait cent sottises pareilles.
Après bien du travail, le coche arrive au haut.
Respirons maintenant ! dit la mouche aussitôt :
J'ai tant fait que nos gens sont enfin dans la plaine.
Çà, messieurs les chevaux, payez-moi de ma peine.
Ainsi certaines gens, faisant les empressés,
S'introduisent dans les affaires :
Ils font partout les nécessaires,
Et, partout importuns, devraient être chassés.


LE SAVETIER ET LE FINANCIER
Un savetier chantait du matin jusqu'au soir :
C'était merveille de le voir,
Merveille de l'ouïr ; il faisait des passages :
Plus content qu'aucun des sept sages.
Son voisin, au contraire, étant tout cousu d'or,
Chantait peu, dormait moins encor :
C'était un homme de finance.
Si sur le point du jour parfois il sommeillait,
Le savetier alors en chantant l'éveillait ;
Et le financier se plaignait
Que les soins de la Providence
N'eussent pas au marché fait vendre le dormir,
Comme le manger et le boire.
En son hôtel il fait venir
Le chanteur, et lui dit : Or çà, sire Grégoire,
Que gagnez-vous par an ? Par an ! ma foi, monsieur
Dit avec un ton de rieur
Le gaillard savetier, ce n'est point ma manière
De compter de la sorte ; et je n'entasse guère
Un jour sur l'autre : il suffit qu'à la fin
J'attrape le bout de l'année ;
Chaque jour amène son pain. -
Eh bien ! que gagnez-vous, dites-moi, par journée ?
Tantôt plus, tantôt moins : le mal est que toujours
(Et sans cela nos gains seraient assez honnêtes),
Le mal est que dans l'an s'entremêlent des jours
Qu'il faut chômer ; on nous ruine en fêtes :
L'une fait tort à l'autre ; et monsieur le curé
De quelque nouveau saint charge toujours son prône.
Le financier, riant de sa naïveté,
Lui dit : Je vous veux mettre aujourd'hui sur le trône.
Prenez ces cent écus ; gardez-les avec soin,
Pour vous en servir au besoin.
Le savetier crut voir tout l'argent que la terre
Avait depuis plus de cent ans,
Produit pour l'usage des gens.
Il retourne chez lui : dans sa cave il enserre
L'argent, et sa joie à la fois.
Plus de chant : il perdit la voix
Du moment qu'il gagna ce qui cause nos peines.
Le sommeil quitta son logis :
Il eut pour hôtes les soucis,
Les soupçons, les alarmes vaines.
Tout le jour il avait l'œil au guet ; et la nuit,
Si quelque chat faisait du bruit,
Le chat prenait l'argent. À la fin le pauvre homme
S'en courut chez celui qu'il ne réveillait plus :
Rendez-moi, lui dit-il, mes chansons et mon somme ;
Et reprenez vos cent écus.


LES ANIMAUX MALADES DE LA PESTE
Un mal qui répand la terreur,
Mal que le ciel en sa fureur
Inventa pour punir les crimes de la terre,
La peste (puisqu'il faut l'appeler par son nom),
Capable d'enrichir en un jour l'Achéron,
Faisait aux animaux la guerre.
Ils ne mouraient pas tous, mais tous étaient frappés :
On n'en voyait point d'occupés
À chercher le soutien d'une mourante vie ;
Nul mets n'excitait leur envie ;
Ni loups ni renards n'épiaient
La douce et l'innocente proie ;
Les tourterelles se fuyaient :
Plus d'amour, partant plus de joie.
Le lion tint conseil, et dit : Mes chers amis,
Je crois que le ciel a permis
Pour nos péchés cette infortune.
Que le plus coupable de nous
Se sacrifie aux traits du céleste courroux ;
Peut-être il obtiendra la guérison commune.
L'histoire nous apprend qu'en de tels accidents
On fait de pareils dévouements.
Ne nous flattons donc point ; voyons sans indulgence
L'état de notre conscience.
Pour moi, satisfaisant mes appétits gloutons,
J'ai dévoré force moutons.
Que m'avaient-ils fait ? nulle offense ;
Même il m'est arrivé quelquefois de manger
Le berger.
Je me dévouerai donc, s'il le faut : mais je pense
Qu'il est bon que chacun s'accuse ainsi que moi ;
Car on doit souhaiter, selon toute justice,
Que le plus coupable périsse.
Sire, dit le renard, vous êtes trop bon roi ;
Vos scrupules font voir trop de délicatesse.
Eh bien ! manger moutons, canaille, sotte espèce,
Est-ce un péché ? Non, non. Vous leur fîtes, seigneur,
En les croquant, beaucoup d'honneur ;
Et quant au berger, l'on peut dire
Qu'il était digne de tous maux,
Étant de ces gens-là qui sur les animaux
Se font un chimérique empire.
Ainsi dit le renard ; et flatteurs d'applaudir.
On n'osa trop approfondir
Du tigre, ni de l'ours, ni des autres puissances,
Les moins pardonnables offenses :
Tous les gens querelleurs, jusqu'aux simples mâtins,
Au dire de chacun, étaient de petits saints.
L'âne vint à son tour, et dit : J'ai souvenance
Qu'en un pré de moines passant,
La faim, l'occasion, l'herbe tendre, et, je pense,
Quelque diable aussi me poussant,
Je tondis de ce pré la largeur de ma langue ;
Je n'en avais nul droit, puisqu'il faut parler net.
À ces mots, on cria haro sur le baudet.
Un loup, quelque peu clerc, prouva par sa harangue
Qu'il fallait dévouer ce maudit animal,
Ce pelé, ce galeux, d'où venait tout leur mal.
Sa peccadille fut jugée un cas pendable.
Manger l'herbe d'autrui ! quel crime abominable !
Rien que la mort n'était capable
D'expier son forfait. On le lui fit bien voir.
Selon que vous serez puissant ou misérable,
Les jugements de cour vous rendront blanc ou noir.


LA POULE AUX ŒUFS D'OR
L'avarice perd tout en voulant tout gagner.
Je ne veux, pour le témoigner,
Que celui dont la poule, à ce que dit la fable,
Pondait tous les jours un œuf d'or.
Il crut que, dans son corps, elle avait un trésor ;
Il la tua, l'ouvrit, et la trouva semblable
À celle dont les œufs ne lui rapportaient rien,
S'étant lui-même ôté le plus beau de son bien.
Belle leçon pour les gens chiches !
Pendant ces derniers temps combien en a-t-on vus
Qui du soir au matin sont pauvres devenus
Pour vouloir trop tôt être riches !


L'OURS ET LES DEUX COMPAGNONS
Deux compagnons, pressés d'argent,
À leur voisin fourreur vendirent
La peau d'un ours encor vivant,
Mais qu'ils tueraient bientôt, du moins à ce qu'ils dirent,
C'était le roi des ours au compte de ces gens.
Le marchand à sa peau devait faire fortune ;
Elle garantirait des froids les plus cuisants ;
On en pourrait fourrer plutôt deux robes qu'une.
Dindenaut prisait moins ses moutons qu'eux leur ours :
Leur, à leur compte, et non à celui de la bête.
S'offrant de la livrer au plus tard dans deux jours,
Ils conviennent de prix, et se mettent en quête,
Trouvent l'ours qui s'avance et vient vers eux au trot,
Voilà mes gens frappés comme d'un coup de foudre.
Le marché ne tint pas ; il fallut le résoudre :
D'intérêts contre l'ours, on n'en dit pas un mot.
L'un des deux compagnons grimpe au faîte d'un arbre ;
L'autre, plus froid que n'est un marbre,
Se couche sur le nez, fait le mort, tient son vent,
Ayant quelque part ouï dire
Que l'ours s'acharne peu souvent
Sur un corps qui ne vit, ne meut, ni ne respire.
Seigneur ours, comme un sot, donna dans ce panneau :
Il voit ce corps gisant, le croit privé de vie ;
Et, de peur de supercherie,
Le tourne, le retourne, approche son museau,
Flaire aux passages de l'haleine.
C'est, dit-il, un cadavre ; ôtons-nous, car il sent.
À ces mots, l'ours s'en va dans la forêt prochaine.
L'un de nos deux marchands de son arbre descend,
Court à son compagnon, lui dit que c'est merveille
Qu'il n'ait eu seulement que la peur pour tout mal.
Eh bien ! ajouta-t-il, la peau de l'animal ?
Mais que t'a-t-il dit à l'oreille ?
Car il t'approchait de bien près,
Te retournant avec sa serre.
Il m'a dit qu'il ne faut jamais
Vendre la peau de l'ours qu'on ne l'ait mis par terre.


LE RENARD ET LE BOUC
Capitaine renard allait de compagnie
Avec son ami bouc des plus haut encornés :
Celui-ci ne voyait pas plus loin que son nez ;
L'autre était passé maître en fait de tromperie.
La soif les obligea de descendre en un puits ;
Là chacun d'eux se désaltère.
Après qu'abondamment tous deux en eurent pris,
Le renard dit au bouc : Que ferons-nous, compère ?
Ce n'est pas tout de boire, il faut sortir d'ici.
Lève tes pieds en haut, et tes cornes aussi ;
Mets-les contre le mur : le long de ton échine
Je grimperai premièrement ;
Puis sur tes cornes m'élevant,
À l'aide de cette machine,
De ce lieu-ci je sortirai,
Après quoi je t'en tirerai.
Par ma barbe, dit l'autre, il est bon ; et je loue
Les gens bien sensés comme toi.
Je n'aurais jamais, quant à moi,
Trouvé ce secret, je l'avoue.
Le renard sort du puits, laisse son compagnon,
Et vous lui fait un beau sermon
Pour l'exhorter à patience.
Si le ciel t'eût, dit-il, donné par excellence
Autant de jugement que de barbe au menton,
Tu n'aurais pas, à la légère,
Descendu dans ce puits. Or, adieu ; j'en suis hors :
Tâche de t'en tirer et fais tous les efforts ;
Car, pour moi, j'ai certaine affaire
Qui ne me permet pas d'arrêter en chemin.
En toute chose il faut considérer la fin.


LE CERF SE VOYANT DANS L'EAU
Dans le cristal d'une fontaine
Un cerf se mirant autrefois
Louait la beauté de son bois,
Et ne pouvait qu'avecque peine
Souffrir ses jambes de fuseaux,
Dont il voyait l'objet se perdre dans les eaux.
Quelle proportion de mes pieds à ma tête !
Disait-il en voyant leur ombre avec douleur :
Des taillis les plus hauts mon front atteint le faîte ;
Mes pieds ne me font point d'honneur.
Tout en parlant de la sorte,
Un limier le fait partir.
Il tâche à se garantir ;
Dans les forêts il s'emporte :
Son bois, dommageable ornement,
L'arrêtant à chaque moment,
Nuit à l'office que lui rendent
Ses pieds de qui ses jours dépendent.
Il se dédit alors, et maudit les présents
Que le ciel lui fait tous les ans.
Nous faisons cas du beau, nous méprisons l'utile ;
Et le beau souvent nous détruit.
Ce cerf blâme ses pieds qui le rendent agile ;
Il estime un bois qui lui nuit.


LE LOUP DEVENU BERGER
Un loup qui commençait d'avoir petite part
Aux brebis de son voisinage,
Crut qu'il fallait s'aider de la peau du renard,
Et faire un nouveau personnage
Il s'habille en berger, endosse un hoqueton,
Fait sa houlette d'un bâton,
Sans oublier la cornemuse.
Pour pousser jusqu'au bout la ruse,
Il aurait volontiers écrit sur son chapeau :
"C'est moi qui suis Guillot, berger de ce troupeau."
Sa personne étant ainsi faite,
Et ses pieds de devant posés sur sa houlette,
Guillot le sycophante approche doucement.
Guillot, le vrai Guillot, étendu sur l'herbette,
Dormait alors profondément ;
Son chien dormait aussi, comme aussi sa musette :
La plupart des brebis dormaient pareillement.
L'hypocrite les laissa faire ;
Et, pour pouvoir mener vers son fort les brebis,
Il voulut ajouter la parole aux habits,
Chose qu'il croyait nécessaire ;
Mais cela gâta son affaire :
Il ne put du pasteur contrefaire la voix.
Le ton dont il parla fit retentir les bois,
Et découvrit tout le mystère.
Chacun se réveille à ce son,
Les brebis, le chien, le garçon.
Le pauvre loup, dans cet esclandre,
Empêché par son hoqueton,
Ne put ni fuir ni se défendre.
Toujours par quelque endroit fourbes se laissent prendre
Quiconque est loup agisse en loup ;
C'est le plus certain de beaucoup.


LE RAT DE VILLE, ET LE RAT DES CHAMPS
Autrefois le rat de ville
Invita le rat des champs,
D'une façon fort civile,
À des reliefs d'ortolans.
Sur un tapis de Turquie
Le couvert se trouva mis.
Je laisse à penser la vie
Que firent ces deux amis.
Le régal fut fort honnête ;
Rien ne manquait au festin :
Mais quelqu'un troubla la fête
Pendant qu'ils étaient en train.
À la porte de la salle
Ils entendirent du bruit :
Le rat de ville détale ;
Son camarade le suit.
Le bruit cesse, on se retire :
Rats en campagne aussitôt ;
Et le citadin de dire :
Achevons tout notre rôt.
C'est assez, dit le rustique :
Demain vous viendrez chez moi.
Ce n'est pas que je me pique
De tous vos festins de roi :
Mais rien ne vient m'interrompre ;
Je mange tout à loisir.
Adieu donc. Fi du plaisir
Que la crainte peut corrompre !


LE PETIT POISSON ET LE PÊCHEUR
Petit poisson deviendra grand,
Pourvu que Dieu lui prête vie ;
Mais le lâcher en attendant,
Je tiens pour moi que c'est folie,
Car de le rattraper il n'est pas trop certain.
Un carpeau qui n'était encore que fretin,
Fut pris par un pêcheur au bord d'une rivière.
Tout fait nombre, dit l'homme, en voyant son butin ;
Voilà commencement de chère et de festin :
Mettons-le en notre gibecière.
Le pauvre carpillon lui dit en sa manière :
Que ferez-vous de moi ? je ne saurais fournir
Au plus qu'une demi-bouchée.
Laissez-moi carpe devenir :
Je serai par vous repêchée ;
Quelque gros partisan m'achètera bien cher :
Au lieu qu'il vous en faut chercher
Peut-être encor cent de ma taille
Pour faire un plat : quel plat ! croyez-moi, rien qui vaille.
Rien qui vaille ! eh bien ! soit, repartit le pêcheur ;
Poisson, mon bel ami, qui faites le prêcheur,
Vous irez dans la poêle ; et, vous avez beau dire,
Dès ce soir on vous fera frire.
Un Tiens, vaut, ce dit-on, mieux que deux Tu l'auras :
L'un est sûr, l'autre ne l'est pas.


LE POT DE TERRE ET LE POT DE FER
Le pot de fer proposa
Au pot de terre un voyage.
Celui-ci s'en excusa,
Disant qu'il ferait que sage
De garder le coin du feu :
Car il lui fallait si peu,
Si peu que la moindre chose
De son débris serait cause :
Il n'en reviendrait morceau.
Pour vous, dit-il, dont la peau
Est plus dure que la mienne,
Je ne vois rien qui vous tienne.
Nous vous mettrons à couvert,
Repartit le pot de fer :
Si quelque matière dure
Vous menace d'aventure,
Entre deux je passerai,
Et du coup vous sauverai.
Cette offre le persuade.
Pot de fer son camarade
Se met droit à ses côtés.
Mes gens s'en vont à trois pieds
Clopin clopant comme ils peuvent,
L'un contre l'autre jetés
Au moindre hoquet qu'ils treuvent.
Le pot de terre en souffre ; il n'eut pas fait cent pas
Que par son compagnon il fut mis en éclats,
Sans qu'il eût lieu de se plaindre.
Ne nous associons qu'avecque nos égaux ;
Ou bien il nous faudra craindre
Le destin d'un de ces pots.


LE LABOUREUR ET SES ENFANTS
Travaillez, prenez de la peine :
C'est le fonds qui manque le moins.
Un riche laboureur, sentant sa mort prochaine,
Fit venir ses enfants, leur parla sans témoins.
Gardez-vous, leur dit-il, de vendre l'héritage
Que nous ont laissé nos parents :
Un trésor est caché dedans.
Je ne sais pas l'endroit ; mais un peu de courage
Vous le fera trouver : vous en viendrez à bout.
Remuez votre champ dès qu'on aura fait l'oût :
Creusez, fouillez, bêchez ; ne laissez nulle place
Où la main ne passe et repasse.
Le père mort, les fils vous retournent le champ,
De çà, de là, partout ; si bien qu'au bout de l'an
Il en rapporta davantage.
D'argent, point de caché. Mais le père fut sage
De leur montrer, avant sa mort,
Que le travail est un trésor.


LE MEUNIER, SON FILS, ET L'ÂNE
L'invention des arts étant un droit d'aînesse,
Nous devons l'apologue à l'ancienne Grèce :
Mais ce champ ne se peut tellement moissonner
Que les derniers venus n'y trouvent à glaner.
La feinte est un pays plein de terres désertes ;
Tous les jours nos auteurs y font des découvertes.
Je t'en veux dire un trait assez bien inventé :
Autrefois à Racan Malherbe l'a conté.
Ces deux rivaux d'Horace, héritiers de sa lyre,
Disciples d'Apollon, nos maîtres, pour mieux dire,
Se rencontrant un jour tout seuls et sans témoins
(Comme ils se confiaient leurs pensers et leurs soins),
Racan commence ainsi : Dites-moi, je vous prie,
Vous qui devez savoir les choses de la vie,
Qui par tous ses degrés avez déjà passé,
Et que rien ne doit fuir en cet âge avancé,
À quoi me résoudrai-je ? Il est temps que j'y pense.
Vous connaissez mon bien, mon talent, ma naissance :
Dois-je dans la province établir mon séjour,
Prendre emploi dans l'armée, ou bien charge à la cour ?
Tout au monde est mêlé d'amertume et de charmes :
La guerre a ses douceurs, l'hymen a ses alarmes.
Si je suivais mon goût, je saurais où buter ;
Mais j'ai les miens, la cour, le peuple à contenter.
Malherbe là-dessus : Contenter tout le monde !
Écoutez ce récit avant que je réponde.
J'ai lu dans quelque endroit qu'un meunier et son fils,
L'un vieillard, l'autre enfant, non pas des plus petits,
Mais garçon de quinze ans, si j'ai bonne mémoire,
Allaient vendre leur âne, un certain jour de foire.
Afin qu'il fût plus frais et de meilleur débit,
On lui lia les pieds, on vous le suspendit ;
Puis cet homme et son fils le portent comme un lustre.
Pauvres gens ! idiots ! couple ignorant et rustre !
Le premier qui les vit de rire s'éclata :
Quelle farce, dit-il, vont jouer ces gens-là ?
Le plus âne des trois n'est pas celui qu'on pense.
Le meunier, à ces mots, connaît son ignorance ;
Il met sur pieds sa bête, et la fait détaler.
L'âne, qui goûtait fort l'autre façon d'aller,
Se plaint en son patois. Le meunier n'en a cure,
Il fait monter son fils, il suit : et, d'aventure,
Passent trois bons marchands. Cet objet leur déplut.
Le plus vieux au garçon s'écria tant qu'il put :
Oh là ! oh ! descendez, que l'on ne vous le dise,
Jeune homme, qui menez laquais à barbe grise !
C'était à vous de suivre, au vieillard de monter.
Messieurs, dit le meunier, il vous faut contenter.
L'enfant met pied à terre, et puis le vieillard monte ;
Quand trois filles passant, l'une dit : C'est grand'honte
Qu'il faille voir ainsi clocher ce jeune fils,
Tandis que ce nigaud, comme un évêque assis,
Fait le veau sur son âne, et pense être bien sage.
Il n'est, dit le meunier, plus de veaux à mon âge :
Passez votre chemin, la fille, et m'en croyez.
Après maints quolibets coup sur coup renvoyés,
L'homme crut avoir tort, et mit son fils en croupe.
Au bout de trente pas, une troisième troupe
Trouve encore à gloser. L'un dit : Ces gens sont fous !
Le baudet n'en peut plus ; il mourra sous leurs coups.
Eh quoi ! charger ainsi cette pauvre bourrique !
N'ont-ils point de pitié de leur vieux domestique ?
Sans doute qu'à la foire ils vont vendre sa peau.
Parbleu ! dit le meunier, est bien fou du cerveau
Qui prétend contenter tout le monde et son père.
Essayons toutefois si par quelque manière
Nous en viendrons à bout. Ils descendent tous deux.
L'âne se prélassant marche seul devant eux.
Un quidam les rencontre et dit : Est-ce la mode
Que baudet aille à l'aise, et meunier s'incommode ?
Qui de l'âne ou du maître est fait pour se lasser ?
Je conseille à ces gens de le faire enchâsser.
Ils usent leurs souliers, et conservent leur âne !
Nicolas, au rebours : car, quand il va voir Jeanne,
Il monte sur sa bête ; et la chanson le dit.
Beau trio de baudets ! le meunier repartit :
Je suis âne, il est vrai, j'en conviens, je l'avoue ;
Mais que dorénavant on me blâme, on me loue,
Qu'on dise quelque chose ou qu'on ne dise rien,
J'en veux faire à ma tête. Il le fit, et fit bien.
Quant à vous, suivez Mars, ou l'Amour, ou le prince ;
Allez, venez, courez ; demeurez en province ;
Prenez femme, abbaye, emploi, gouvernement :
Les gens en parleront, n'en doutez nullement.


LE CHAT, LA BELETTE, ET LE PETIT LAPIN
Du palais d'un jeune lapin
Dame belette, un beau matin,
S'empara : c'est une rusée.
Le maître étant absent, ce lui fut chose aisée.
Elle porta chez lui ses pénates, un jour
Qu'il était allé faire à l'Aurore sa cour
Parmi le thym et la rosée.
Après qu'il eut brouté, trotté, fait tous ses tours,
Jeannot lapin retourne aux souterrains séjours.
La belette avait mis le nez à la fenêtre.
Ô dieux hospitaliers ! que vois-je ici paraître ?
Dit l'animal chassé du paternel logis.
Holà ! madame la belette,
Que l'on déloge sans trompette,
Ou je vais avertir tous les rats du pays.
La dame au nez pointu répondit que la terre
Était au premier occupant.
C'était un beau sujet de guerre,
Qu'un logis où lui-même il n'entrait qu'en rampant !
Et quand ce serait un royaume,
Je voudrais bien savoir, dit-elle, quelle loi
En a pour toujours fait l'octroi
À Jean, fils ou neveu de Pierre ou de Guillaume,
Plutôt qu'à Paul, plutôt qu'à moi.
Jean lapin allégua la coutume et l'usage :
Ce sont, dit-il, leurs lois qui m'ont de ce logis
Rendu maître et seigneur, et qui, de père en fils,
L'ont de Pierre à Simon, puis à moi Jean, transmis.
Le premier occupant, est-ce une loi plus sage ?
Or bien, sans crier davantage,
Rapportons-nous, dit-elle, à Raminagrobis.
C'était un chat vivant comme un dévot ermite,
Un chat faisant la chattemite,
Un saint homme de chat, bien fourré, gros et gras,
Arbitre expert sur tous les cas.
Jean lapin pour juge l'agrée.
Les voilà tous deux arrivés
Devant sa majesté fourrée.
Grippeminaud leur dit : Mes enfants, approchez,
Approchez : je suis sourd, les ans en sont la cause.
L'un et l'autre approcha, ne craignant nulle chose.
Aussitôt qu'à portée il vit les contestants,
Grippeminaud le bon apôtre,
Jetant des deux côtés la griffe en même temps,
Mit les plaideurs d'accord en croquant l'un et l'autre.
Ceci ressemble fort aux débats qu'ont parfois
Les petits souverains se rapportant aux rois.


LES DEUX MULETS
Deux mulets cheminaient, l'un d'avoine chargé,
L'autre portant l'argent de la gabelle.
Celui-ci, glorieux d'une charge si belle,
N'eût voulu pour beaucoup en être soulagé.
Il marchait d'un pas relevé
Et faisait sonner sa sonnette ;
Quand l'ennemi se présentant,
Comme il en voulait à l'argent,
Sur le mulet du fisc une troupe se jette,
Le saisit au frein, et l'arrête.
Le mulet, en se défendant,
Se sent percer de coups ; il gémit, il soupire.
Est-ce donc là, dit-il, ce qu'on m'avait promis ?
Ce mulet qui me suit du danger se retire,
Et moi j'y tombe et je péris !
Ami, lui dit son camarade,
Il n'est pas toujours bon d'avoir un haut emploi :
Si tu n'avais servi qu'un meunier comme moi,
Tu ne serais pas si malade.
"""

print(f"{len(corpus)} caractères")
print(corpus[:236])

38331 caractères
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêt


In [3]:
# Le tokeniseur caractère du chapitre 10... plus TROIS tokens spéciaux.
# Un token spécial est une balise réservée : il ne correspond à aucun caractère du texte,
# il sert de panneau de signalisation ("ici parle l'utilisateur", "ici répond l'assistant").
# On les stocke comme des caractères de contrôle invisibles (x01, x02, x03),
# et on les affiche sous une forme lisible au décodage.
SPECIAUX = {"<|user|>": "\x01", "<|assistant|>": "\x02", "<|fin|>": "\x03"}

chars = sorted(set(corpus)) + sorted(SPECIAUX.values())
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}          # caractère -> numéro
itos = {i: c for c, i in stoi.items()}              # numéro -> caractère
encode = lambda s: [stoi[c] for c in s]

def decode(ids):
    s = "".join(itos[i] for i in ids)
    for nom, ch in SPECIAUX.items():                # rendre les tokens spéciaux lisibles
        s = s.replace(ch, nom)
    return s

USR = stoi["\x01"]       # id du token <|user|>
ASS = stoi["\x02"]       # id du token <|assistant|>
FIN = stoi["\x03"]       # id du token <|fin|>
print(f"vocabulaire : {vocab_size} tokens (81 caractères des fables + 3 tokens spéciaux)")

vocabulaire : 84 tokens (81 caractères des fables + 3 tokens spéciaux)


In [4]:
# Le GPT du chapitre 10, tel quel. Seul changement : block_size passe de 64 à 128,
# pour qu'une paire question + réponse entière tienne dans le contexte.
block_size = 128

def decouper_en_tetes(x, n_heads):
    B, T, d_model = x.shape
    return x.view(B, T, n_heads, d_model // n_heads).transpose(1, 2)   # (B, h, T, d_k)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        Q = decouper_en_tetes(self.W_Q(x), self.n_heads)
        K = decouper_en_tetes(self.W_K(x), self.n_heads)
        V = decouper_en_tetes(self.W_V(x), self.n_heads)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float('-inf'))
        poids = torch.softmax(scores, dim=-1)
        out = (poids @ V).transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


d_model, n_heads, n_layers, d_ff = 96, 4, 2, 384


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


def nb_params(m):
    return sum(p.numel() for p in m.parameters())

On pré-entraîne le modèle de base sur les fables (2 500 pas, comme au chapitre 16), puis on mesure sa perplexité sur les fables : c'est notre **thermomètre**. On s'en servira pour détecter l'oubli catastrophique.

In [5]:
# Pré-entraînement : prédire le caractère suivant sur les 30 fables (chapitre 10, tel quel).
data_fables = torch.tensor(encode(corpus))

def fabriquer_batch(data, taille=32):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), décalé de 1
    return x, y

debut = time.time()
torch.manual_seed(42)
gpt_base = GPT()
opt = torch.optim.AdamW(gpt_base.parameters(), lr=3e-3)
for step in range(2500):
    x, y = fabriquer_batch(data_fables)
    loss = F.cross_entropy(gpt_base(x).view(-1, vocab_size), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()

etat_base = copy.deepcopy(gpt_base.state_dict())   # on garde une copie des poids du modèle de base
print(f"modèle de base : {nb_params(gpt_base)} paramètres, entraîné en {time.time() - debut:.0f} s (loss {loss.item():.2f})")

modèle de base : 251520 paramètres, entraîné en 76 s (loss 0.24)


In [6]:
# La perplexité du chapitre 16, telle quelle : notre thermomètre "fables".
@torch.no_grad()
def loss_moyenne(model, data):
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for i in range((len(data) - 1) // block_size):
        x = data[i * block_size : (i + 1) * block_size].unsqueeze(0)
        y = data[i * block_size + 1 : (i + 1) * block_size + 1].unsqueeze(0)
        nll = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1), reduction='sum')
        total_nll += nll.item(); total_tokens += y.numel()
    model.train()
    return total_nll / total_tokens

def perplexite(model, data):
    return math.exp(loss_moyenne(model, data))

ppl_base = perplexite(gpt_base, data_fables)
print(f"perplexité du modèle de base sur les fables : {ppl_base:.2f}")

perplexité du modèle de base sur les fables : 1.27


## 1. Le fossé : il complète, il n'obéit pas

Le modèle de base fait UNE chose : prédire le caractère suivant. Donne-lui un début de fable, il continue la fable. Donne-lui une question au format dialogue, il... continue avec ce qu'il connaît : de la fable. Il ne répond pas, il complète.

In [7]:
@torch.no_grad()
def generer(model, ids, n=100, temperature=0.8, seed=0, stop=True):
    torch.manual_seed(seed)
    model.eval()
    ids = list(ids)
    for _ in range(n):
        x = torch.tensor([ids[-block_size:]])
        probas = torch.softmax(model(x)[0, -1] / temperature, dim=-1)
        suivant = torch.multinomial(probas, 1).item()
        ids.append(suivant)
        if stop and suivant == FIN:      # en usage normal, on s'arrête au token <|fin|>
            break
    model.train()
    return ids

def prompt_ids(question):
    return [USR] + encode(question) + [ASS]

# Ce qu'il sait faire : compléter une fable.
print(decode(generer(gpt_base, encode("La cigale, ayant"), n=80, seed=1)))

La cigale, ayant graissez me égal.
Avec nait dit leur âne, un cette étant vicir :
Je suis âne, i


In [8]:
# Ce qu'il ne sait PAS faire : répondre à une question.
for q in ["Quelle est la capitale du Bénin ?", "Combien font deux plus deux ?"]:
    print(decode(generer(gpt_base, prompt_ids(q), n=80, seed=1)))
    print("-" * 60)

<|user|>Quelle est la capitale du Bénin ?<|assistant|>u sole n'est rale, dit-elle,
Avant l'ocerf l'oût,
Que le partient corbeau, hu si
------------------------------------------------------------


<|user|>Combien font deux plus deux ?<|assistant|>néris mon âne,
L'onteau pers pencor vous en fraisent et d'œil nous à mourrontre 
------------------------------------------------------------


## 2. Le dataset d'instructions : 60 paires écrites à la main

Pas de dataset téléchargé ici : on écrit nos 60 paires (question, réponse) à la main, en français, pour voir chaque exemple qui entre dans le modèle. Capitales, calcul en toutes lettres, contraires, pluriels, un peu de fables. C'est minuscule (un vrai SFT en compte des dizaines de milliers), mais le mécanisme est identique.

In [9]:
# 60 paires (instruction, réponse), écrites à la main.
# Contrainte assumée : uniquement des caractères que notre tokeniseur connaît
# (pas de chiffres ni de k/w : les fables n'en contiennent pas, les nombres sont en lettres).
PAIRES = [
    ('Quelle est la capitale du Bénin ?', 'La capitale du Bénin est Porto-Novo.'),
    ('Quelle est la capitale du Togo ?', 'La capitale du Togo est Lomé.'),
    ('Quelle est la capitale du Niger ?', 'La capitale du Niger est Niamey.'),
    ('Quelle est la capitale du Nigéria ?', 'La capitale du Nigéria est Abuja.'),
    ('Quelle est la capitale de la France ?', 'La capitale de la France est Paris.'),
    ('Quelle est la capitale du Ghana ?', 'La capitale du Ghana est Accra.'),
    ('Combien font deux plus deux ?', 'Deux plus deux font quatre.'),
    ('Combien font trois plus cinq ?', 'Trois plus cinq font huit.'),
    ('Combien font dix moins quatre ?', 'Dix moins quatre font six.'),
    ('Combien font six fois sept ?', 'Six fois sept font quarante-deux.'),
    ('Combien font neuf plus huit ?', 'Neuf plus huit font dix-sept.'),
    ('Combien font cinq fois cinq ?', 'Cinq fois cinq font vingt-cinq.'),
    ('Combien font huit plus sept ?', 'Huit plus sept font quinze.'),
    ('Qui a écrit la fable du Corbeau et du Renard ?', 'Cette fable a été écrite par Jean de La Fontaine.'),
    ('Donne le contraire du mot grand.', 'Le contraire de grand est petit.'),
    ('Donne le contraire du mot chaud.', 'Le contraire de chaud est froid.'),
    ('Donne le contraire du mot rapide.', 'Le contraire de rapide est lent.'),
    ('Donne le contraire du mot ouvert.', 'Le contraire de ouvert est fermé.'),
    ('Donne le contraire du mot jour.', 'Le contraire de jour est nuit.'),
    ('Traduis le mot maison en anglais.', 'Le mot maison se traduit par house.'),
    ('Traduis le mot soleil en anglais.', 'Le mot soleil se traduit par sun.'),
    ('Traduis le mot chat en anglais.', 'Le mot chat se traduit par cat.'),
    ('Comment dit-on bonjour en anglais ?', 'On dit hello en anglais.'),
    ('Comment dit-on oui en anglais ?', 'On dit yes en anglais.'),
    ('Une mangue coûte cent francs CFA. Combien coûtent deux mangues ?', 'Deux mangues coûtent deux cents francs CFA.'),
    ('Sètondji parcourt cinq bornes à zémidjan puis trois de plus. Combien en tout ?', 'Sètondji parcourt huit bornes en tout.'),
    ('Trois sacs de riz coûtent neuf cents francs CFA. Combien coûte un sac ?', 'Un sac de riz coûte trois cents francs CFA.'),
    ('Épelle le mot chat.', "Le mot chat s'épelle c, h, a, t."),
    ('Épelle le mot ami.', "Le mot ami s'épelle a, m, i."),
    ('Donne le pluriel du mot cheval.', 'Le pluriel de cheval est chevaux.'),
    ('Donne le pluriel du mot journal.', 'Le pluriel de journal est journaux.'),
    ('Donne le pluriel du mot travail.', 'Le pluriel de travail est travaux.'),
    ('Quel jour vient après lundi ?', 'Après lundi vient mardi.'),
    ('Quel jour vient avant dimanche ?', 'Avant dimanche vient samedi.'),
    ('Quel mois vient après juin ?', 'Après juin vient juillet.'),
    ('Quel mois vient avant mars ?', 'Avant mars vient février.'),
    ('Combien de jours compte une semaine ?', 'Une semaine compte sept jours.'),
    ('Combien de mois compte une année ?', 'Une année compte douze mois.'),
    ('Quelle est la couleur du ciel par beau temps ?', 'Par beau temps, le ciel est bleu.'),
    ("Quelle est la couleur de l'herbe ?", "L'herbe est verte."),
    ('Nomme un fruit tropical.', 'La mangue est un fruit tropical.'),
    ('Nomme un animal de la ferme.', 'La poule est un animal de la ferme.'),
    ("Nomme un animal qui vit dans l'eau.", "Le poisson vit dans l'eau."),
    ('Que boit-on quand on a soif ?', "Quand on a soif, on boit de l'eau."),
    ('Que mange une chèvre ?', "Une chèvre mange de l'herbe."),
    ('À quoi sert un parapluie ?', 'Un parapluie sert à se protéger de la pluie.'),
    ('À quoi sert une lampe ?', 'Une lampe sert à éclairer.'),
    ('À quoi sert un couteau ?', 'Un couteau sert à couper.'),
    ('Complète : rien ne sert de courir, il faut...', 'Il faut partir à point.'),
    ('Complète : petit poisson deviendra...', 'Petit poisson deviendra grand.'),
    ('Complète : tout flatteur vit aux dépens...', "Tout flatteur vit aux dépens de celui qui l'écoute."),
    ('Donne un synonyme du mot content.', 'Un synonyme de content est heureux.'),
    ('Donne un synonyme du mot vite.', 'Un synonyme de vite est rapidement.'),
    ('Donne un synonyme du mot maison.', 'Un synonyme de maison est demeure.'),
    ('Combien de pattes a une araignée ?', 'Une araignée a huit pattes.'),
    ('Combien de pattes a un chien ?', 'Un chien a quatre pattes.'),
    ('Combien de doigts compte une main ?', 'Une main compte cinq doigts.'),
    ("Quel animal chante tout l'été dans la fable ?", "C'est la cigale qui chante tout l'été."),
    ('Quel animal tient un fromage dans son bec ?', "C'est le corbeau qui tient un fromage."),
    ("Quel est le premier mois de l'année ?", "Le premier mois de l'année est janvier."),
]
print(f"{len(PAIRES)} paires instruction/réponse")

60 paires instruction/réponse


## 3. Le template et le masquage de la loss

Chaque paire devient une séquence : `<|user|>` + question + `<|assistant|>` + réponse + `<|fin|>`.

Le point capital, ce sont les **labels**. On veut que le modèle apprenne à générer la **réponse**, pas la question. Chaque token du prompt reçoit donc le label `-100` : la valeur que `F.cross_entropy` ignore par défaut (`ignore_index=-100`). La loss ne compte que sur la réponse et sur `<|fin|>` (le modèle doit aussi apprendre à s'arrêter).

Comme au pré-entraînement, on concatène tous les exemples en un long flux et on y découpe des fenêtres : même `fabriquer_batch`, mêmes shapes, seule la colonne des labels change.

In [10]:
def fabriquer_flux_sft(paires, masquer=True, n_epoques=6, seed=0):
    """Concatène les exemples (mélangés, répétés) en un long flux, comme au pré-entraînement.
    Retourne deux tenseurs alignés : les ids, et les labels (avec -100 sur ce qu'on ne veut pas apprendre)."""
    rng = random.Random(seed)
    ids_flux, labels_flux = [], []
    for _ in range(n_epoques):
        melange = paires[:]
        rng.shuffle(melange)
        for question, reponse in melange:
            prompt = [USR] + encode(question) + [ASS]        # la partie "consigne"
            cible = encode(reponse) + [FIN]                  # la partie "à apprendre"
            ids_flux += prompt + cible
            if masquer:
                labels_flux += [-100] * len(prompt) + cible  # prompt masqué : -100
            else:
                labels_flux += prompt + cible                # tout compte (on verra que c'est un bug)
    return torch.tensor(ids_flux), torch.tensor(labels_flux)

ids_sft, labels_sft = fabriquer_flux_sft(PAIRES, masquer=True)
n_masques = (labels_sft == -100).sum().item()
print(f"flux SFT : {len(ids_sft)} tokens, dont {n_masques} masqués ({100 * n_masques / len(ids_sft):.0f} %)")

flux SFT : 24738 tokens, dont 12990 masqués (53 %)


In [11]:
# Validation du masquage sur un exemple minuscule.
ids_t, labels_t = fabriquer_flux_sft([("Deux ?", "Trois.")], masquer=True, n_epoques=1)
n_prompt = 1 + len("Deux ?") + 1                # <|user|> + question + <|assistant|>
assert len(ids_t) == len(labels_t), "ids et labels doivent être alignés position par position"
assert labels_t[:n_prompt].eq(-100).all().item(), "tout le prompt (tokens spéciaux inclus) doit être à -100"
assert labels_t[n_prompt:].eq(ids_t[n_prompt:]).all().item(), "la réponse (et <|fin|>) doit garder ses ids"
assert labels_t[-1].item() == FIN, "le dernier label doit être le token <|fin|> : le modèle doit apprendre à s'arrêter"
print("masquage OK : prompt à -100, réponse + <|fin|> conservés")

masquage OK : prompt à -100, réponse + <|fin|> conservés


In [12]:
# À quoi ressemble le flux ? Un exemple décodé, et l'alignement ids/labels sur ses premiers tokens.
apercu, apercu_labels = fabriquer_flux_sft([PAIRES[0]], masquer=True, n_epoques=1)
print(decode(apercu.tolist()))
print()
print(f"{'position':>9} | {'entrée':>15} | label")
print("-" * 42)
for t in range(0, len(apercu) - 1):
    if t < 4 or abs(t - (len(apercu) - 1 - len(PAIRES[0][1]) - 1)) < 3 or t > len(apercu) - 4:
        entree = decode([apercu[t].item()])
        label = apercu_labels[t + 1].item()          # à la position t, on prédit le token t+1
        affiche = "-100 (ignoré)" if label == -100 else repr(decode([label]))
        print(f"{t:>9} | {entree!r:>15} | {affiche}")
    elif t == 5:
        print(f"{'...':>9} |")

<|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|>

 position |          entrée | label
------------------------------------------
        0 |      '<|user|>' | -100 (ignoré)
        1 |             'Q' | -100 (ignoré)
        2 |             'u' | -100 (ignoré)
        3 |             'e' | -100 (ignoré)
      ... |
       32 |             ' ' | -100 (ignoré)
       33 |             '?' | -100 (ignoré)
       34 | '<|assistant|>' | 'L'
       35 |             'L' | 'a'
       36 |             'a' | ' '
       69 |             'o' | '.'
       70 |             '.' | '<|fin|>'


In [13]:
def sft_batch(ids, labels, taille=16):
    ix = torch.randint(0, len(ids) - block_size - 1, (taille,))
    x = torch.stack([ids[i : i + block_size] for i in ix])              # (B, T)
    y = torch.stack([labels[i + 1 : i + block_size + 1] for i in ix])   # (B, T), labels décalés de 1
    return x, y

x, y = sft_batch(ids_sft, labels_sft)
print("x :", tuple(x.shape), "| y :", tuple(y.shape), "| labels ignorés dans ce batch :", (y == -100).sum().item())

x : (16, 128) | y : (16, 128) | labels ignorés dans ce batch : 1085


## 4. SFT : l'affinage supervisé

On repart des poids du modèle de base (on ne réinitialise rien) et on continue l'entraînement sur le flux d'instructions, avec la loss masquée. 1 200 pas suffisent : le modèle sait déjà écrire du français de fable, il n'apprend qu'un **comportement**.

In [14]:
# Validation : la loss doit ignorer les positions à -100.
logits_t = torch.zeros(1, 4, vocab_size)
logits_t[0, :, 0] = 10.0                     # le modèle prédit "token 0" partout, très sûr de lui
y_bon    = torch.tensor([[0, 0, 0, 0]])      # labels corrects partout
y_masque = torch.tensor([[0, 0, -100, -100]])# deux positions masquées (labels "faux" ignorés)
l1 = F.cross_entropy(logits_t.view(-1, vocab_size), y_bon.view(-1), ignore_index=-100)
l2 = F.cross_entropy(logits_t.view(-1, vocab_size), y_masque.view(-1), ignore_index=-100)
assert abs(l1.item() - l2.item()) < 1e-6, "les positions -100 doivent être exclues de la moyenne"
print(f"OK : loss identique avec ou sans positions masquées ({l1.item():.6f})")

OK : loss identique avec ou sans positions masquées (0.003761)


In [15]:
def sft(etat_depart, ids, labels, steps=1200, lr=1e-3, seed=42):
    """Repart des poids du modèle de base, et continue l'entraînement sur le flux SFT."""
    torch.manual_seed(seed)
    m = GPT()
    m.load_state_dict(etat_depart)                  # on NE repart PAS de zéro : on affine
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    for step in range(steps):
        x, y = sft_batch(ids, labels)
        logits = m(x)                               # (B, T, vocab)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return m, loss.item()

debut = time.time()
gpt_sft, loss_finale = sft(etat_base, ids_sft, labels_sft)
print(f"SFT (masqué) : {time.time() - debut:.0f} s, loss finale {loss_finale:.3f}")

SFT (masqué) : 25 s, loss finale 0.047


In [16]:
prompts_test = [
    "Quelle est la capitale du Bénin ?",
    "Combien font deux plus deux ?",
    "Donne le contraire du mot grand.",
    "Une mangue coûte cent francs CFA. Combien coûtent deux mangues ?",
]
for q in prompts_test:
    print("AVANT :", decode(generer(gpt_base, prompt_ids(q), n=70, seed=1)))
    print("APRÈS :", decode(generer(gpt_sft,  prompt_ids(q), n=70, seed=1)))
    print("-" * 70)

AVANT : <|user|>Quelle est la capitale du Bénin ?<|assistant|>u sole n'est rale, dit-elle,
Avant l'ocerf l'oût,
Que le partient corb
APRÈS : <|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|>
----------------------------------------------------------------------
AVANT : <|user|>Combien font deux plus deux ?<|assistant|>néris mon âne,
L'onteau pers pencor vous en fraisent et d'œil nous à m


APRÈS : <|user|>Combien font deux plus deux ?<|assistant|>Deux plus deux font quatre.<|fin|>
----------------------------------------------------------------------


AVANT : <|user|>Donne le contraire du mot grand.<|assistant|>ce s'en voloil, ferant ! meune la bête.
Sal lle malher du que c'os s'a
APRÈS : <|user|>Donne le contraire du mot grand.<|assistant|>Le contraire de grand est petit.<|fin|>
----------------------------------------------------------------------


AVANT : <|user|>Une mangue coûte cent francs CFA. Combien coûtent deux mangues ?<|assistant|>n vein tous voir jus ne gagne sa sanse.
Le pauve chicher : il fut vise
APRÈS : <|user|>Une mangue coûte cent francs CFA. Combien coûtent deux mangues ?<|assistant|>Deux mangues coûtent deux cents francs CFA.<|fin|>
----------------------------------------------------------------------


Le modèle obéit sur les questions du dataset. Mais attention : 60 exemples ne font pas une intelligence générale. Posons une question **absente** du dataset (aucune paire ne parle du Mali) :

In [17]:
# Une question ABSENTE du dataset (aucune paire sur le Mali) :
print(decode(generer(gpt_sft, prompt_ids("Quelle est la capitale du Mali ?"), n=70, seed=1)))

<|user|>Quelle est la capitale du Mali ?<|assistant|>La capitale du Ghana est Accra.<|fin|>


## 5. Cas qui échoue n°1 : oublier le masquage

On refait exactement le même SFT, mais avec `masquer=False` : la loss compte sur TOUT, questions comprises. Le modèle apprend donc aussi... à écrire des questions. Pour le voir, on laisse les deux modèles générer au-delà de `<|fin|>` (stop désactivé).

In [18]:
# Le même SFT, mais SANS masquer la loss sur le prompt (masquer=False).
ids_nomask, labels_nomask = fabriquer_flux_sft(PAIRES, masquer=False)
debut = time.time()
gpt_nomask, loss_nomask = sft(etat_base, ids_nomask, labels_nomask)
print(f"SFT (sans masque) : {time.time() - debut:.0f} s, loss finale {loss_nomask:.3f}")

SFT (sans masque) : 26 s, loss finale 0.108


In [19]:
# On laisse chaque modèle générer 150 tokens SANS s'arrêter à <|fin|> (stop=False),
# pour voir ce qu'il a VRAIMENT appris à produire après sa réponse.
q = "Quelle est la capitale du Bénin ?"
print("SANS masque (le bug) :")
print(decode(generer(gpt_nomask, prompt_ids(q), n=150, seed=3, stop=False)))
print()
print("AVEC masque (correct) :")
print(decode(generer(gpt_sft, prompt_ids(q), n=150, seed=3, stop=False))[:260], "...")

SANS masque (le bug) :


<|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|><|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|><|user|>Quelle est la capitale du Togo ?<|assistant|>La capi

AVEC masque (correct) :


<|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|>Para house.<|fin|>n, i.<|fin|>L'herpello qui crut de L'hance vier chevaux.<|fin|>Le coûte contrun vient à étrite par hon anne est P ...


## 6. Cas qui échoue n°2 : l'oubli catastrophique, mesuré

Notre SFT complet a bougé TOUS les poids, avec un learning rate soutenu, sur un seul type de données. Qu'est-il arrivé à ce que le modèle savait avant ? Le thermomètre du chapitre 16 répond : on mesure la perplexité sur les fables, avant et après.

In [20]:
# Le thermomètre "fables" avant / après SFT complet :
ppl_apres_sft = perplexite(gpt_sft, data_fables)
print(f"perplexité fables, modèle de base : {ppl_base:.2f}")
print(f"perplexité fables, après SFT      : {ppl_apres_sft:.2f}")
print()
print("Et si on lui redemande une fable ?")
print(decode(generer(gpt_sft, encode("La cigale, ayant"), n=90, seed=1, stop=False))[:220])

perplexité fables, modèle de base : 1.27
perplexité fables, après SFT      : 81.36

Et si on lui redemande une fable ?
La cigale, ayante par Jean.<|fin|>La parde vient juillet.<|fin|>Le vient février.<|fin|>nne de L'herbe est verte.<|fin|>Le contrai


## 7. LoRA : la correction de rang faible, écrite à la main

Le SFT complet a deux défauts qu'on vient de mesurer : il réécrit tous les poids (l'original est perdu), et il coûte la mémoire d'un entraînement complet (poids + gradients + états Adam pour chaque paramètre).

LoRA gèle la matrice W et apprend une **correction** de rang faible : W reste intacte, la couche calcule `W x + (alpha/r) · B(A x)`, et seules les deux petites matrices A et B s'entraînent. On l'écrit de nos mains, on la pose sur les projections d'attention et la tête, et on compte les paramètres.

In [21]:
class LoRALinear(nn.Module):
    """Enveloppe une couche nn.Linear : W est gelée, on apprend la correction (alpha/r) * B @ A."""

    def __init__(self, couche, r=8, alpha=16.0):
        super().__init__()
        self.couche = couche
        for p in self.couche.parameters():
            p.requires_grad = False                       # W ne bougera plus jamais
        self.r = r
        self.echelle = alpha / r                          # le facteur alpha/r de la formule
        # A : (r, d_entree), petite gaussienne ; B : (d_sortie, r), zéros
        self.A = nn.Parameter(torch.randn(r, couche.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(couche.out_features, r))

    def forward(self, x):
        # x @ A.T : (B, T, d_entree) -> (B, T, r)   : on descend dans le goulot
        # ... @ B.T : (B, T, r) -> (B, T, d_sortie) : on remonte
        return self.couche(x) + (x @ self.A.T @ self.B.T) * self.echelle

    def fusionner(self):
        """Absorbe la correction dans W : W <- W + (alpha/r) * B @ A."""
        with torch.no_grad():
            self.couche.weight.data.add_((self.B @ self.A) * self.echelle)

In [22]:
# À l'initialisation, B = 0 : la correction est nulle, le modèle doit être EXACTEMENT le même.
torch.manual_seed(0)
test_lin = nn.Linear(96, 96, bias=False)
test_lora = LoRALinear(test_lin, r=8, alpha=16.0)
x_test = torch.randn(2, 5, 96)
ecart_init = (test_lora(x_test) - test_lin(x_test)).abs().max().item()
assert ecart_init == 0.0, "avec B = 0, la couche LoRA doit rendre exactement la sortie d'origine"
print(f"OK : à l'init, sortie identique (écart max = {ecart_init})")

OK : à l'init, sortie identique (écart max = 0.0)


In [23]:
def poser_lora(model, r=8, alpha=16.0):
    """Gèle tout le modèle, puis enveloppe les 4 projections d'attention et la tête finale."""
    for p in model.parameters():
        p.requires_grad = False
    for bloc in model.blocs:
        for nom in ("W_Q", "W_K", "W_V", "W_O"):
            setattr(bloc.attn, nom, LoRALinear(getattr(bloc.attn, nom), r, alpha))
    model.tete = LoRALinear(model.tete, r, alpha)
    return model

def retirer_lora(model):
    """Débranche les adaptateurs : chaque LoRALinear rend sa couche d'origine."""
    for bloc in model.blocs:
        for nom in ("W_Q", "W_K", "W_V", "W_O"):
            module = getattr(bloc.attn, nom)
            if isinstance(module, LoRALinear):
                setattr(bloc.attn, nom, module.couche)
    if isinstance(model.tete, LoRALinear):
        model.tete = model.tete.couche
    return model

torch.manual_seed(42)
gpt_lora = GPT()
gpt_lora.load_state_dict(etat_base)                  # on repart du modèle de base
gpt_lora = poser_lora(gpt_lora, r=8, alpha=16.0)

params_lora = [p for p in gpt_lora.parameters() if p.requires_grad]
n_entrainables = sum(p.numel() for p in params_lora)
n_total = sum(p.numel() for p in gpt_lora.parameters())
print(f"paramètres entraînables : {n_entrainables} / {n_total} ({100 * n_entrainables / n_total:.1f} %)")
print(f"taille de l'adaptateur : {n_entrainables * 4 / 1024:.0f} Ko (en float32)")

paramètres entraînables : 13728 / 265248 (5.2 %)
taille de l'adaptateur : 54 Ko (en float32)


In [24]:
debut = time.time()
opt = torch.optim.AdamW(params_lora, lr=2e-3)        # SEULS les A et B reçoivent des mises à jour
for step in range(1500):
    x, y = sft_batch(ids_sft, labels_sft)
    loss = F.cross_entropy(gpt_lora(x).view(-1, vocab_size), y.view(-1), ignore_index=-100)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"SFT LoRA : {time.time() - debut:.0f} s, loss finale {loss.item():.3f}")

for q in prompts_test[:3]:
    print(decode(generer(gpt_lora, prompt_ids(q), n=70, seed=1)))

SFT LoRA : 34 s, loss finale 0.090
<|user|>Quelle est la capitale du Bénin ?<|assistant|>La capitale du Bénin est Porto-Novo.<|fin|>
<|user|>Combien font deux plus deux ?<|assistant|>Deux plus deux font quatre.<|fin|>
<|user|>Donne le contraire du mot grand.<|assistant|>Le contraire de grand est petit.<|fin|>


In [25]:
ppl_lora_branche = perplexite(gpt_lora, data_fables)

gpt_debranche = retirer_lora(copy.deepcopy(gpt_lora))     # on retire les adaptateurs
ppl_lora_debranche = perplexite(gpt_debranche, data_fables)

print(f"perplexité fables, adaptateurs branchés   : {ppl_lora_branche:.2f}")
print(f"perplexité fables, adaptateurs débranchés : {ppl_lora_debranche:.2f}")
print(f"perplexité fables, modèle de base         : {ppl_base:.2f}")
assert abs(ppl_lora_debranche - ppl_base) < 1e-6, "W gelée : débrancher LoRA doit rendre le modèle de base EXACT"

perplexité fables, adaptateurs branchés   : 701.90
perplexité fables, adaptateurs débranchés : 1.27
perplexité fables, modèle de base         : 1.27


In [26]:
# Fusion : on absorbe B @ A dans W, et la sortie doit rester identique (au bruit float près).
x_test = torch.tensor([prompt_ids("Combien font deux plus deux ?")])
gpt_lora.eval()
with torch.no_grad():
    sortie_avant = gpt_lora(x_test)

gpt_fusionne = copy.deepcopy(gpt_lora)
for module in gpt_fusionne.modules():
    if isinstance(module, LoRALinear):
        module.fusionner()
gpt_fusionne = retirer_lora(gpt_fusionne)            # plus aucune couche LoRA : modèle standard
gpt_fusionne.eval()
with torch.no_grad():
    sortie_apres = gpt_fusionne(x_test)

ecart = (sortie_avant - sortie_apres).abs().max().item()
print(f"écart max avant/après fusion : {ecart:.2e}")
assert ecart < 1e-3, "la fusion ne doit pas changer la sortie"

écart max avant/après fusion : 3.24e-05


## 8. QLoRA : l'idée, et un jouet 4 bits

QLoRA pousse la logique un cran plus loin : puisque W est gelée, autant la **stocker compressée**. Les poids de base passent en 4 bits (16 niveaux possibles par poids, plus une échelle par groupe), les adaptateurs LoRA restent en pleine précision. Nul besoin de GPU pour comprendre le mécanisme : on l'implémente en jouet sur nos propres poids et on mesure ce que la compression coûte.

In [27]:
def quantifier_4bit(W, taille_groupe=64):
    """Quantification 4 bits jouet : par groupe de 64 poids, 16 niveaux entiers (-8..7) + une échelle."""
    plat = W.flatten()
    pad = (-len(plat)) % taille_groupe
    plat = torch.cat([plat, torch.zeros(pad)])
    groupes = plat.view(-1, taille_groupe)
    echelles = groupes.abs().max(dim=1, keepdim=True).values / 7.0   # 1 float par groupe
    q = torch.clamp((groupes / echelles).round(), -8, 7).to(torch.int8)
    return q, echelles, W.shape, pad

def dequantifier_4bit(q, echelles, shape, pad):
    plat = (q.float() * echelles).flatten()
    if pad:
        plat = plat[:-pad]
    return plat.view(shape)

W = gpt_base.blocs[0].attn.W_Q.weight.data
q, e, sh, pd = quantifier_4bit(W)
W_reconstruit = dequantifier_4bit(q, e, sh, pd)
print(f"erreur moyenne de reconstruction : {(W - W_reconstruit).abs().mean().item():.5f}")
print(f"amplitude moyenne des poids      : {W.abs().mean().item():.5f}")

erreur moyenne de reconstruction : 0.01470
amplitude moyenne des poids      : 0.12816


In [28]:
# On quantifie-déquantifie TOUTES les couches Linear du modèle de base, et on mesure les dégâts.
gpt_4bit = copy.deepcopy(gpt_base)
n_couches = 0
for module in gpt_4bit.modules():
    if isinstance(module, nn.Linear):
        q, e, sh, pd = quantifier_4bit(module.weight.data)
        module.weight.data = dequantifier_4bit(q, e, sh, pd)
        n_couches += 1
ppl_4bit = perplexite(gpt_4bit, data_fables)
print(f"{n_couches} couches Linear quantifiées en 4 bits")
print(f"perplexité fables : {ppl_base:.2f} (float32) contre {ppl_4bit:.2f} (4 bits)")

13 couches Linear quantifiées en 4 bits
perplexité fables : 1.27 (float32) contre 2.02 (4 bits)


### QLoRA pour de vrai (Colab GPU)

Le vrai QLoRA utilise la bibliothèque `bitsandbytes` (quantification NF4, dont les 16 niveaux suivent les quantiles d'une gaussienne) : elle exige un GPU NVIDIA. La cellule ci-dessous ne s'exécute que sur Colab avec un GPU ; partout ailleurs elle se saute proprement.

In [29]:
# QLoRA réel (Colab GPU uniquement) : cette cellule ne fait rien sans GPU NVIDIA.
# Sur Colab (Exécution > Modifier le type d'exécution > T4), décommente le pip install et exécute.
if torch.cuda.is_available():
    try:
        # !pip install -q bitsandbytes transformers accelerate peft
        from transformers import AutoModelForCausalLM, BitsAndBytesConfig

        config_4bit = BitsAndBytesConfig(
            load_in_4bit=True,                       # poids de base stockés en 4 bits
            bnb_4bit_quant_type="nf4",               # niveaux NF4 (quantiles d'une gaussienne)
            bnb_4bit_use_double_quant=True,          # les échelles sont quantifiées à leur tour
            bnb_4bit_compute_dtype=torch.bfloat16,   # calcul (déquantifié) en bf16
        )
        modele = AutoModelForCausalLM.from_pretrained(
            "HuggingFaceTB/SmolLM2-135M",            # petit modèle ouvert, révision figée conseillée
            quantization_config=config_4bit, device_map="auto",
        )
        print("modèle chargé en NF4 :", sum(p.numel() for p in modele.parameters()), "paramètres")
    except Exception as e:
        print("QLoRA réel non disponible ici :", e)
else:
    print("Pas de GPU CUDA détecté : cellule sautée (c'est prévu, tout le reste du notebook tourne sur CPU).")

Pas de GPU CUDA détecté : cellule sautée (c'est prévu, tout le reste du notebook tourne sur CPU).


## 9. Comme les vrais labos

Ce que tu viens de faire est, en miniature exacte, le post-entraînement d'un vrai labo :

- **OLMo 2** (Ai2) publie tout son SFT : le mélange de données **Tülu 3** (env. 939 000 conversations, publiques sur Hugging Face : `allenai/tulu-3-sft-mixture`), le code, et les poids avant/après chaque étape. Tu peux vérifier chaque ingrédient toi-même.
- Le template y est plus riche (rôle `system`, multi-tour), le masquage est **le même** : loss sur les réponses de l'assistant uniquement.
- LoRA/QLoRA servent partout où l'on affine un modèle sans réécrire tous ses poids : un adaptateur par tâche, une base partagée.

À retenir : le SFT n'ajoute pas de connaissances, il installe un **comportement**. Et il se paie : l'oubli catastrophique se surveille avec des évaluations de rétention, exactement comme notre thermomètre fables.

## Exercices

À toi de jouer : quatre exercices sur les gestes clés du chapitre (les labels masqués, la loss qui ignore le prompt, le forward de LoRA, la fusion de l'adaptateur), du plus simple (●) au plus costaud (●●●). Chaque cellule marquée `# TODO(toi)` contient un trou ; complète-le, puis exécute la cellule de validation (`assert`) qui suit : si elle passe sans erreur, c'est gagné.

Les exercices s'appuient sur les objets de la leçon (`encode`, `USR`, `ASS`, `FIN`, `vocab_size`) : exécute la leçon avant de commencer.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi. Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Les labels masqués — niveau ●

Dans la leçon, `fabriquer_flux_sft` a construit les labels à ta place. Refais le geste sur UNE paire : chaque token du prompt reçoit le label `-100`, chaque token de la réponse (et `<|fin|>`) garde son id.

In [ ]:
def exemple_sft(question, reponse):
    """Rend (ids, labels) pour une paire, deux listes alignées position par position."""
    prompt = [USR] + encode(question) + [ASS]        # la partie "consigne"
    cible = encode(reponse) + [FIN]                  # la partie "à apprendre"
    ids = prompt + cible
    # TODO(toi) : construis les labels masqués.
    # Rappel : la loss ne doit compter QUE sur la réponse (cible).
    # Chaque token du prompt reçoit le label -100 (ignoré par cross_entropy),
    # chaque token de la réponse garde son id comme label.
    labels = ...
    return ids, labels

In [ ]:
# Validation : les labels masqués.
ids_t, labels_t = exemple_sft("Deux ?", "Trois.")
n_prompt = 1 + len("Deux ?") + 1                     # <|user|> + question + <|assistant|>
assert len(labels_t) == len(ids_t), "ids et labels doivent être alignés position par position"
assert labels_t[:n_prompt] == [-100] * n_prompt, "tout le prompt (tokens spéciaux inclus) doit être à -100"
assert labels_t[n_prompt:] == ids_t[n_prompt:], "la réponse (et <|fin|>) doit garder ses ids"
assert labels_t[-1] == FIN, "le dernier label doit être <|fin|> : s'arrêter fait partie de la leçon"
print("Labels OK : prompt à -100, réponse + <|fin|> conservés.")

### Exercice 2 · La loss qui ignore le prompt — niveau ●

La fonction `sft` de la leçon tient sur une ligne de loss. Réécris-la : une cross-entropy qui exclut toutes les positions dont le label vaut `-100`.

In [ ]:
def loss_sft(logits, labels):
    """Cross-entropy du SFT : les positions dont le label vaut -100 ne comptent pas."""
    # TODO(toi) : calcule la cross-entropy en IGNORANT les labels -100.
    # Indice : logits.view(-1, vocab_size), labels.view(-1), et un argument de
    # F.cross_entropy dont la valeur par défaut est justement -100.
    return ...

In [ ]:
# Validation : la loss doit ignorer les positions à -100.
torch.manual_seed(1)
logits_t = torch.randn(2, 6, vocab_size)
labels_t = torch.randint(0, vocab_size, (2, 6))
labels_t[:, :3] = -100                               # la moitié des positions est masquée
garde = labels_t.view(-1) != -100
l_ref = F.cross_entropy(logits_t.view(-1, vocab_size)[garde], labels_t.view(-1)[garde])
l_exo = loss_sft(logits_t, labels_t)
assert torch.isclose(l_exo, l_ref, atol=1e-6), f"la loss doit valoir {l_ref.item():.6f}, pas {l_exo.item():.6f}"
print(f"Loss OK : {l_exo.item():.6f}, moyenne sur les seules positions non masquées.")

### Exercice 3 · Le forward de LoRA — niveau ●●

La classe ci-dessous reprend l'enveloppe de la leçon : W gelée, A en petite gaussienne, B à zéro. Il te manque le forward : la sortie de la couche gelée, plus la correction de rang faible à l'échelle alpha/r. Surveille tes shapes à chaque étape.

In [ ]:
class LoRALinearExo(nn.Module):
    """Enveloppe une couche nn.Linear : W est gelée, on apprend la correction (alpha/r) * B @ A."""

    def __init__(self, couche, r=8, alpha=16.0):
        super().__init__()
        self.couche = couche
        for p in self.couche.parameters():
            p.requires_grad = False                   # W ne bougera plus jamais
        self.echelle = alpha / r                      # le facteur alpha/r de la formule
        # A : (r, d_entree), petite gaussienne ; B : (d_sortie, r), zéros
        self.A = nn.Parameter(torch.randn(r, couche.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(couche.out_features, r))

    def forward(self, x):
        # TODO(toi) : la sortie LoRA = sortie de la couche gelée + correction de rang faible.
        # 1) sortie gelée : self.couche(x)
        # 2) correction : x @ A.T (descente vers r dims) puis @ B.T (remontée), fois self.echelle
        # Vérifie les shapes : x est (B, T, d_entree), la correction doit finir en (B, T, d_sortie).
        return ...

In [ ]:
# Validation : le forward de LoRA.
torch.manual_seed(0)
lin_t = nn.Linear(96, 64, bias=False)                # une couche rectangulaire, exprès
lora_t = LoRALinearExo(lin_t, r=8, alpha=16.0)
x_t = torch.randn(2, 5, 96)
assert lora_t(x_t).shape == (2, 5, 64), f"shape obtenue : {tuple(lora_t(x_t).shape)}, attendue : (2, 5, 64)"
assert (lora_t(x_t) - lin_t(x_t)).abs().max().item() == 0.0, "avec B = 0, la sortie doit être EXACTEMENT celle de la couche gelée"
with torch.no_grad():
    lora_t.B.copy_(torch.randn_like(lora_t.B) * 0.1)  # on branche une correction non nulle
attendu = lin_t(x_t) + F.linear(x_t, lora_t.B @ lora_t.A) * lora_t.echelle   # Delta W = B @ A
ecart = (lora_t(x_t) - attendu).abs().max().item()
assert ecart < 1e-5, f"écart {ecart:.2e} : la correction doit valoir (alpha/r) * B(Ax)"
print("LoRA forward OK : correction nulle à l'init, formule exacte ensuite.")

### Exercice 4 · Fusionner l'adaptateur — niveau ●●●

Dernier geste du chapitre : absorber la correction dans W, une fois pour toutes, pour servir le modèle sans le détour de calcul LoRA. Ta fonction reçoit une couche LoRA de l'exercice 3 (la validation réutilise ta classe `LoRALinearExo`) et doit modifier `couche.weight` en place. L'ordre du produit compte.

In [ ]:
def fusionner(lora):
    """Absorbe la correction dans W : W <- W + (alpha/r) * B @ A (modifie lora.couche en place)."""
    with torch.no_grad():
        # TODO(toi) : ajoute la correction aux poids de lora.couche, SANS oublier le
        # facteur d'échelle. Attention à l'ordre du produit : B @ A a la shape
        # (d_sortie, d_entree), celle de W ; A @ B ne colle même pas.
        ...

In [ ]:
# Validation : la fusion.
torch.manual_seed(0)
lin_t = nn.Linear(96, 96, bias=False)
lora_t = LoRALinearExo(lin_t, r=8, alpha=16.0)
with torch.no_grad():
    lora_t.B.copy_(torch.randn_like(lora_t.B) * 0.1)
x_t = torch.randn(2, 5, 96)
avant = lora_t(x_t)                                  # sortie avec adaptateur branché
fusionner(lora_t)
apres = lora_t.couche(x_t)                           # la couche seule, correction absorbée
ecart = (avant - apres).abs().max().item()
assert ecart < 1e-4, f"écart {ecart:.2e} : la couche fusionnée doit rendre la même sortie que LoRA"
print(f"Fusion OK : écart max {ecart:.2e}, la correction est absorbée dans W.")

## Verdict

Quatre validations vertes : tu sais masquer des labels, calculer la loss qui ignore le prompt, écrire le forward de LoRA et fusionner un adaptateur. Autrement dit, les gestes du post-entraînement supervisé, tu les as dans les doigts.

Ton GPT des fables sait désormais obéir. Mais relis sa réponse sur le Mali : obéir n'est pas savoir. Entre deux réponses bien formées, comment apprendre au modèle à préférer la meilleure ? C'est le chapitre 18, « Apprendre des préférences », la deuxième moitié de l'éducation.